# M4 new — Metric-Guided Sequential Training## Core ideaInstead of fixed orderings (M2), **adapt the training objective after each stage** based on which capability is currently weakest — measured by validation loss decomposition.## Algorithm1. **Stage 1**: Always start with E7 (decision-only) — Phase 1 showed it builds the strongest foundation2. **Evaluate**: Compute val loss on decision tokens vs explanation tokens3. **Select next loss**: If explanation loss > decision loss → train with E6 (explanation-only). Otherwise → train with E1 (WSFT for overall quality).4. **Stage 3**: Always finish with E5b (explanation entropy) — adds safety calibration as a final polish5. Each stage: 1 epoch, decreasing LR## Why this works- Stage 1 is always E7 because Phase 1 proved it's the best foundation builder- Stage 2 adapts to the model's specific weakness after Stage 1- Stage 3 is always entropy matching because it acts as a regularizer and doesn't disrupt the model- Total: 3 epochs (same budget as M2)

In [ ]:
# Cell 0: Environment + Model Selection (Linux/WSL2)
import os, torch

TRAIN_MODEL = "qwen25_1p5b_base"
#TRAIN_MODEL = "qwen25_3b_base"
#TRAIN_MODEL = "qwen25_7b_base"

use_bf16 = "7b" not in TRAIN_MODEL

print(f"GPU: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print(f"Will train: {TRAIN_MODEL} | bf16={use_bf16}")

In [ ]:
# Cell 1: Imports + data
import os, sys, json, math, re
from typing import List, Dict, Any, Tuple
from tqdm import tqdm

import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import TrainingArguments, Trainer
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

sys.path.insert(0, os.path.expanduser("~/kd_project"))
from shared_utils import (
    RUNS_DIR, MAX_SEQ_LEN, LR, SEED, LAMBDA,
    W_FORMAT, W_DECISION, W_EXPL,
    STUDENTS, load_data, load_student,
    get_section_spans, in_any_span,
    compute_mean_confidence,
    find_decision_span_chars, find_expl_span_chars,
    teacher_section_entropy_mean,
    build_prompt_text, tokenize_and_mask, FlexibleCollator,
)

OUT_DIR = os.path.join(RUNS_DIR, "m4_metric_guided")
os.makedirs(OUT_DIR, exist_ok=True)

STAGE_LR = {0: 2e-4, 1: 7e-5, 2: 2e-5}
EPOCHS_PER_STAGE = 1

prompts, teacher_rows = load_data()
MEAN_C = compute_mean_confidence(teacher_rows)

# Small val set for metric evaluation
import random
rng = random.Random(SEED)
indices = list(range(len(teacher_rows)))
rng.shuffle(indices)
val_indices = set(indices[:200])
train_rows_mg = [teacher_rows[i] for i in indices if i not in val_indices]
val_rows_mg = [teacher_rows[i] for i in val_indices]
print(f"Metric-guided train: {len(train_rows_mg)}, val: {len(val_rows_mg)}")

In [ ]:
# Cell 2: Dataset — precomputes all fields
class M2v2DatasetFast(Dataset):
    def __init__(self, rows, prompts, tokenizer, is_instruct, mean_c):
        print("  Precomputing dataset...")
        self.items = []
        for idx in tqdm(range(len(rows)), desc="  Tokenizing"):
            r = rows[idx]
            prompt = prompts[r["id"]]
            answer = r["teacher_text"]

            prompt_text = build_prompt_text(tokenizer, prompt, is_instruct)
            input_ids, offsets, labels, answer_start = tokenize_and_mask(
                tokenizer, prompt_text, answer
            )

            # WSFT weights
            wsft_weights = [0.0] * len(input_ids)
            spans = get_section_spans(answer)
            dec_spans = [(answer_start + s, answer_start + e) for s, e in spans["decision"]]
            expl_spans = [(answer_start + s, answer_start + e) for s, e in spans["explanation"]]
            for i, (s, e) in enumerate(offsets):
                if e <= s: continue
                if s >= answer_start:
                    w = W_FORMAT
                    if in_any_span(s, e, dec_spans): w = W_DECISION
                    if in_any_span(s, e, expl_spans): w = W_EXPL
                    wsft_weights[i] = float(w)
            active_w = [w for w in wsft_weights if w > 0]
            if active_w:
                mw = sum(active_w) / len(active_w)
                if mw > 1e-6:
                    wsft_weights = [w / mw if w > 0 else 0.0 for w in wsft_weights]

            # Decision mask
            dec_mask = torch.zeros(len(input_ids), dtype=torch.bool)
            dec_span_chars = find_decision_span_chars(answer)
            if dec_span_chars:
                ds, de = dec_span_chars
                for i, (s, e) in enumerate(offsets):
                    if labels[i] != -100 and not (e <= answer_start + ds or s >= answer_start + de):
                        dec_mask[i] = True

            # Explanation mask
            expl_mask = torch.zeros(len(input_ids), dtype=torch.bool)
            expl_span_chars = find_expl_span_chars(answer)
            if expl_span_chars:
                es, ee = expl_span_chars
                for i, (s, e) in enumerate(offsets):
                    if labels[i] != -100 and not (e <= answer_start + es or s >= answer_start + ee):
                        expl_mask[i] = True

            # Teacher entropy on explanation
            ent_teacher_expl = teacher_section_entropy_mean(r, expl_span_chars)

            self.items.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "wsft_weights": torch.tensor(wsft_weights, dtype=torch.float),
                "dec_mask": dec_mask,
                "expl_mask": expl_mask,
                "ent_teacher_expl": ent_teacher_expl.float(),
            })
        print(f"  ✅ Precomputed {len(self.items)} samples")

    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

print("✅ Dataset class ready")

In [ ]:
# Cell 3: Three single-objective trainers
class DecOnlyTrainer(Trainer):
    """E7-style: loss on decision tokens only."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._ce = torch.nn.CrossEntropyLoss(reduction="none")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        _ = inputs.pop("wsft_weights"); _ = inputs.pop("expl_mask"); _ = inputs.pop("ent_teacher_expl")
        dec_mask = inputs.pop("dec_mask")

        logits = model(**inputs).logits
        sl = logits[:, :-1, :].contiguous()
        ll = labels[:, 1:].contiguous()
        dm = dec_mask[:, 1:]
        V = sl.size(-1)

        tok_loss = self._ce(sl.view(-1, V), ll.view(-1)).view(ll.size())
        active = (ll != -100).float()
        da = dm.float() * active
        loss = (tok_loss * da).sum() / da.sum().clamp(min=1.0)
        return (loss, model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])) if return_outputs else loss


class WsftTrainer(Trainer):
    """E1-style: section-weighted SFT."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._ce = torch.nn.CrossEntropyLoss(reduction="none")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        wsft_w = inputs.pop("wsft_weights")
        _ = inputs.pop("dec_mask"); _ = inputs.pop("expl_mask"); _ = inputs.pop("ent_teacher_expl")

        logits = model(**inputs).logits
        sl = logits[:, :-1, :].contiguous()
        ll = labels[:, 1:].contiguous()
        sw = wsft_w[:, 1:].contiguous()
        V = sl.size(-1)

        tok_loss = self._ce(sl.view(-1, V), ll.view(-1)).view(ll.size())
        active = (ll != -100).float()
        w = sw * active
        denom = active.sum(dim=1).clamp(min=1.0)
        loss = ((tok_loss * w).sum(dim=1) / denom).mean()
        return (loss, model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])) if return_outputs else loss


class ExplEntropyTrainer(Trainer):
    """E5b-style: explanation entropy matching + base SFT."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._ce = torch.nn.CrossEntropyLoss(reduction="none")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        wsft_w = inputs.pop("wsft_weights")
        _ = inputs.pop("dec_mask")
        expl_mask = inputs.pop("expl_mask")
        ent_teacher = inputs.pop("ent_teacher_expl")

        logits = model(**inputs).logits
        sl = logits[:, :-1, :].contiguous()
        ll = labels[:, 1:].contiguous()
        sw = wsft_w[:, 1:].contiguous()
        em = expl_mask[:, 1:]
        B, S, V = sl.size()

        tok_loss = self._ce(sl.view(-1, V), ll.view(-1)).view(B, S)
        active = (ll != -100).float()

        # Base SFT component (keeps format)
        w = sw * active
        denom = active.sum(dim=1).clamp(min=1.0)
        L_sft = ((tok_loss * w).sum(dim=1) / denom).mean()

        # Entropy matching on explanation tokens
        expl_active = em & (ll != -100)
        ent_student = torch.zeros(B, device=sl.device)
        for b in range(B):
            idx = expl_active[b].nonzero(as_tuple=True)[0]
            if len(idx) > 0:
                p = torch.softmax(sl[b, idx, :], dim=-1)
                ent_student[b] = -(p * torch.log(p + 1e-9)).sum(-1).mean()
        L_ent = LAMBDA * ((ent_student - ent_teacher.to(sl.device)) ** 2).mean()

        loss = L_sft + L_ent
        return (loss, model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])) if return_outputs else loss


TRAINER_MAP = {
    "dec": DecOnlyTrainer,
    "wsft": WsftTrainer,
    "ent": ExplEntropyTrainer,
}
LOSS_DISPLAY = {"dec": "L_dec_only (E7)", "wsft": "L_wsft (E1)", "ent": "L_expl_ent (E5b)"}

print("✅ All three trainers ready")

In [ ]:
# Cell 4: Metric-guided training loop
import gc

cfg = STUDENTS[TRAIN_MODEL]
out_path = os.path.join(OUT_DIR, TRAIN_MODEL)
os.makedirs(out_path, exist_ok=True)

# Precompute datasets
tokenizer_ref = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
if tokenizer_ref.pad_token is None: tokenizer_ref.pad_token = tokenizer_ref.eos_token
train_ds = M2v2DatasetFast(train_rows_mg, prompts, tokenizer_ref, cfg["is_instruct"], MEAN_C)
val_ds = M2v2DatasetFast(val_rows_mg, prompts, tokenizer_ref, cfg["is_instruct"], MEAN_C)
collator = FlexibleCollator(
    tokenizer_ref,
    extra_1d_fields=["wsft_weights", "dec_mask", "expl_mask"],
    extra_scalar_fields=["ent_teacher_expl"],
)


@torch.no_grad()
def evaluate_section_losses(model, val_dataset, collator):
    """Returns (dec_loss, expl_loss) — avg CE on decision vs explanation tokens."""
    model.eval()
    loader = DataLoader(val_dataset, batch_size=1, collate_fn=collator, shuffle=False)
    dec_sum, dec_count = 0.0, 0
    expl_sum, expl_count = 0.0, 0
    ce = torch.nn.CrossEntropyLoss(reduction="none")

    for batch in loader:
        batch = {k: v.to("cuda") if hasattr(v, "to") else v for k, v in batch.items()}
        labels = batch.pop("labels"); _ = batch.pop("wsft_weights")
        dm = batch.pop("dec_mask"); em = batch.pop("expl_mask"); _ = batch.pop("ent_teacher_expl")
        logits = model(**batch).logits
        sl = logits[:, :-1, :]; ll = labels[:, 1:]
        dm2 = dm[:, 1:]; em2 = em[:, 1:]
        V = sl.size(-1)
        tl = ce(sl.reshape(-1, V), ll.reshape(-1)).view(ll.size())
        active = (ll != -100).float()

        da = dm2.float() * active
        if da.sum() > 0:
            dec_sum += (tl * da).sum().item()
            dec_count += da.sum().item()

        ea = em2.float() * active
        if ea.sum() > 0:
            expl_sum += (tl * ea).sum().item()
            expl_count += ea.sum().item()

    model.train()
    dec_loss = dec_sum / max(1, dec_count)
    expl_loss = expl_sum / max(1, expl_count)
    return dec_loss, expl_loss


# ── Stage 1: Always E7 (decision-only) ──
stage1_dir = os.path.join(out_path, "stage1_dec")
os.makedirs(stage1_dir, exist_ok=True)

if os.path.exists(os.path.join(stage1_dir, "adapter_config.json")):
    print("⏩ Stage 1 already done")
else:
    print(f"\n── Stage 1: L_dec_only (E7) | LR={STAGE_LR[0]:.1e} ──")
    tokenizer, model = load_student(TRAIN_MODEL, cfg)
    trainer = DecOnlyTrainer(
        model=model,
        args=TrainingArguments(
            output_dir=stage1_dir, num_train_epochs=EPOCHS_PER_STAGE,
            per_device_train_batch_size=8, gradient_accumulation_steps=4,
            learning_rate=STAGE_LR[0], logging_steps=25, save_strategy="no",
            bf16=use_bf16, seed=SEED, report_to="none",
            remove_unused_columns=False, dataloader_num_workers=4,
            label_smoothing_factor=0.1, gradient_checkpointing="7b" in TRAIN_MODEL,
        ),
        train_dataset=train_ds, data_collator=collator,
    )
    trainer.train()
    model.save_pretrained(stage1_dir)
    tokenizer.save_pretrained(stage1_dir)
    print(f"  ✅ Stage 1 saved")
    del model, tokenizer, trainer; gc.collect(); torch.cuda.empty_cache()

# ── Stage 2: Adaptive — evaluate and choose ──
print("\n── Evaluating after Stage 1 to choose Stage 2 loss... ──")
tokenizer = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
if "7b" in TRAIN_MODEL:
    from transformers import BitsAndBytesConfig
    bnb = BitsAndBytesConfig(load_in_8bit=True)
    base = AutoModelForCausalLM.from_pretrained(cfg["path"], quantization_config=bnb, device_map="auto", trust_remote_code=True)
else:
    base = AutoModelForCausalLM.from_pretrained(cfg["path"], torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base, stage1_dir, is_trainable=True)
model.train()

dec_loss, expl_loss = evaluate_section_losses(model, val_ds, collator)
print(f"  Decision loss: {dec_loss:.4f}")
print(f"  Explanation loss: {expl_loss:.4f}")

if expl_loss > dec_loss:
    stage2_key = "expl_only"
    stage2_name = "stage2_expl"
    TrainerClass2 = WsftTrainer  # Use WSFT which covers explanations with higher weight
    print(f"  → Explanation is weaker. Stage 2: L_wsft (E1) to improve explanation + overall")
else:
    stage2_key = "wsft"
    stage2_name = "stage2_wsft"
    TrainerClass2 = WsftTrainer
    print(f"  → Decision is weaker. Stage 2: L_wsft (E1) for balanced improvement")

stage2_dir = os.path.join(out_path, stage2_name)
os.makedirs(stage2_dir, exist_ok=True)

if os.path.exists(os.path.join(stage2_dir, "adapter_config.json")):
    print("⏩ Stage 2 already done")
    prev_dir = stage2_dir
else:
    print(f"\n── Stage 2: {stage2_name} | LR={STAGE_LR[1]:.1e} ──")
    trainer = TrainerClass2(
        model=model,
        args=TrainingArguments(
            output_dir=stage2_dir, num_train_epochs=EPOCHS_PER_STAGE,
            per_device_train_batch_size=8, gradient_accumulation_steps=4,
            learning_rate=STAGE_LR[1], logging_steps=25, save_strategy="no",
            bf16=use_bf16, seed=SEED+1, report_to="none",
            remove_unused_columns=False, dataloader_num_workers=4,
            label_smoothing_factor=0.1, gradient_checkpointing="7b" in TRAIN_MODEL,
        ),
        train_dataset=train_ds, data_collator=collator,
    )
    trainer.train()
    model.save_pretrained(stage2_dir)
    tokenizer.save_pretrained(stage2_dir)
    print(f"  ✅ Stage 2 saved")
    prev_dir = stage2_dir

    del trainer; gc.collect(); torch.cuda.empty_cache()

# ── Stage 3: Always E5b (entropy calibration) ──
stage3_dir = os.path.join(out_path, "stage3_ent")
os.makedirs(stage3_dir, exist_ok=True)

if os.path.exists(os.path.join(stage3_dir, "adapter_config.json")):
    print("⏩ Stage 3 already done")
else:
    # Reload from stage 2 if model was freed
    if "model" not in dir() or model is None:
        if "7b" in TRAIN_MODEL:
            base = AutoModelForCausalLM.from_pretrained(cfg["path"], quantization_config=bnb, device_map="auto", trust_remote_code=True)
        else:
            base = AutoModelForCausalLM.from_pretrained(cfg["path"], torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
        model = PeftModel.from_pretrained(base, prev_dir, is_trainable=True)
        model.train()

    print(f"\n── Stage 3: L_expl_ent (E5b) | LR={STAGE_LR[2]:.1e} ──")
    trainer = ExplEntropyTrainer(
        model=model,
        args=TrainingArguments(
            output_dir=stage3_dir, num_train_epochs=EPOCHS_PER_STAGE,
            per_device_train_batch_size=8, gradient_accumulation_steps=4,
            learning_rate=STAGE_LR[2], logging_steps=25, save_strategy="no",
            bf16=use_bf16, seed=SEED+2, report_to="none",
            remove_unused_columns=False, dataloader_num_workers=4,
            label_smoothing_factor=0.1, gradient_checkpointing="7b" in TRAIN_MODEL,
        ),
        train_dataset=train_ds, data_collator=collator,
    )
    trainer.train()
    model.save_pretrained(stage3_dir)
    tokenizer.save_pretrained(stage3_dir)
    print(f"  ✅ Stage 3 saved")

# Log the full run
run_log = {
    "model": TRAIN_MODEL,
    "stage1": "dec_only (E7)",
    "stage2_choice": stage2_name,
    "dec_loss_after_s1": dec_loss,
    "expl_loss_after_s1": expl_loss,
    "stage3": "expl_entropy (E5b)",
}
with open(os.path.join(out_path, "run_log.json"), "w") as f:
    json.dump(run_log, f, indent=2)

print(f"\n✅ M4 Metric-Guided complete for {TRAIN_MODEL}")
print(f"   Chose: E7 → {stage2_name} → E5b")